# 23 — CNN + LSTM (Hybrid)

**Architecture:** Embedding → Conv1D(64, k=5) → MaxPool(4) → LSTM(64) → Dense(64, relu) → Dropout(0.3) → Dense(1, sigmoid)  
**Features:** Token sequences (max 200 tokens, vocab 30K)  
The hybrid CNN+LSTM extracts local n-gram features with convolution, then uses LSTM to model the temporal dependencies across those feature maps — combining strengths of both architectures.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, plot_confusion_matrix, plot_roc_curve, plot_pr_curve)
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report)
BASE_DIR, FIGURES_DIR = setup_paths()
import tensorflow as tf
print(f'TensorFlow {tf.__version__}')

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)

MAX_WORDS, MAX_LEN, MAX_SAMPLES = 30000, 200, 80000
if len(X_train) > MAX_SAMPLES:
    np.random.seed(42)
    idx = np.random.choice(len(X_train), MAX_SAMPLES, replace=False)
    X_tr_raw, y_tr = X_train[idx], y_train[idx]
else:
    X_tr_raw, y_tr = X_train, y_train

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_tr_raw)
X_tr = pad_sequences(tokenizer.texts_to_sequences(X_tr_raw), maxlen=MAX_LEN, padding='post', truncating='post')
X_te = pad_sequences(tokenizer.texts_to_sequences(X_test),   maxlen=MAX_LEN, padding='post', truncating='post')
vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
print(f'Vocabulary: {vocab_size:,}  |  Train: {len(X_tr):,}  |  Test: {len(X_te):,}')

## Model Architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

MODEL_NAME = 'CNN + LSTM'
model = Sequential([
    Embedding(vocab_size, 128, input_length=MAX_LEN),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=4),
    LSTM(64, dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='CNN_LSTM')
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
t0 = time.time()
history = model.fit(X_tr, y_tr, epochs=10, batch_size=64,
                    validation_split=0.1, callbacks=[early_stop], verbose=1)
train_time = time.time() - t0
print(f'\nTraining completed in {train_time:.1f}s')

## Training Convergence

In [ ]:
hist = history.history
epochs_run = range(1, len(hist['loss']) + 1)
best_ep = np.argmin(hist['val_loss']) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_run, hist['loss'],     '#3498db', lw=2, label='Train Loss')
axes[0].plot(epochs_run, hist['val_loss'], '#e74c3c', lw=2, linestyle='--', label='Val Loss')
axes[0].axvline(best_ep, color='green', linestyle=':', lw=1.5, label=f'Best={best_ep}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('CNN+LSTM — Loss Convergence')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_run, hist['accuracy'],     '#3498db', lw=2, label='Train Accuracy')
axes[1].plot(epochs_run, hist['val_accuracy'], '#e74c3c', lw=2, linestyle='--', label='Val Accuracy')
axes[1].axvline(best_ep, color='green', linestyle=':', lw=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('CNN+LSTM — Accuracy per Epoch')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('CNN + LSTM — Training Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cnn_lstm_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
y_prob = model.predict(X_te, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_test, y_prob):.4f}')
print(f'Train Time: {train_time:.1f}s')
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test==0], bins=50, alpha=0.65, color='#e74c3c', label='Fake', density=True)
ax.hist(y_prob[y_test==1], bins=50, alpha=0.65, color='#3498db', label='Real', density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5)
ax.set_xlabel('P(Real)'); ax.set_ylabel('Density')
ax.set_title('CNN + LSTM — Prediction Score Distribution')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cnn_lstm_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()